<a href="https://colab.research.google.com/github/amancsecoer/computervisionlab/blob/main/Labsheet3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("/content/titanic_cleaned.csv")

print("Dataset shape:", df.shape)
print(df.head())

In [ ]:
df.columns = df.columns.str.strip()

print("Shape of dataset:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
data = df.copy()

if 'age' in data.columns:
    data['age'] = data['age'].fillna(data['age'].median())

if 'fare' in data.columns:
    data['fare'] = data['fare'].fillna(data['fare'].median())

if 'embarked' in data.columns:
    data['embarked'] = data['embarked'].fillna(
        data['embarked'].mode()[0]
    )

print("Missing values after cleaning:")
print(data.isnull().sum())

In [ ]:
numeric_columns = ['age', 'fare', 'sibsp', 'parch']

print("MEAN")
print(data[numeric_columns].mean())

print("\nMEDIAN")
print(data[numeric_columns].median())

print("\nMODE")
print(data[numeric_columns].mode().iloc[0])

print("\nVARIANCE")
print(data[numeric_columns].var())

print("\nSTANDARD DEVIATION")
print(data[numeric_columns].std())

print("\nCOMPLETE DESCRIPTIVE STATISTICS")
print(data[numeric_columns].describe())

In [ ]:
print("Survival Count:")
print(data['survived'].value_counts())

plt.figure(figsize=(6, 4))

sns.countplot(x='survived', data=data)

plt.title("Titanic Survival Count")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Number of Passengers")

plt.show()

In [ ]:
print("Survival by Gender:")

gender_survival = pd.crosstab(
    data['sex'],
    data['survived']
)

print(gender_survival)

plt.figure(figsize=(7, 5))

sns.countplot(
    x='sex',
    hue='survived',
    data=data
)

plt.title("Survival by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Passengers")

plt.show()

In [ ]:
correlation_columns = [
    'survived',
    'pclass',
    'age',
    'sibsp',
    'parch',
    'fare'
]

correlation_matrix = data[correlation_columns].corr()

print("Correlation Matrix:")
print(correlation_matrix)

plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Titanic Correlation Matrix")
plt.show()

In [ ]:
correlation, p_value = stats.pearsonr(
    data['age'],
    data['fare']
)

print("Pearson Correlation:", correlation)
print("P-value:", p_value)

In [ ]:
survived_age = data[data['survived'] == 1]['age']

not_survived_age = data[data['survived'] == 0]['age']

t_statistic, t_pvalue = stats.ttest_ind(
    survived_age,
    not_survived_age,
    equal_var=False
)

print("T-TEST RESULTS")
print("----------------------")
print("T-statistic:", t_statistic)
print("P-value:", t_pvalue)

alpha = 0.05

if t_pvalue < alpha:
    print("\nDecision: Reject H0")
    print("There is a significant difference in average age.")
else:
    print("\nDecision: Fail to Reject H0")
    print("There is no significant difference in average age.")

In [ ]:
class_1 = data[data['pclass'] == 1]['fare']

class_2 = data[data['pclass'] == 2]['fare']

class_3 = data[data['pclass'] == 3]['fare']

f_statistic, anova_pvalue = stats.f_oneway(
    class_1,
    class_2,
    class_3
)

print("ANOVA RESULTS")
print("----------------------")
print("F-statistic:", f_statistic)
print("P-value:", anova_pvalue)

alpha = 0.05

if anova_pvalue < alpha:
    print("\nDecision: Reject H0")
    print("There is a significant difference in fare among passenger classes.")
else:
    print("\nDecision: Fail to Reject H0")
    print("There is no significant difference in fare among passenger classes.")

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    x='pclass',
    y='fare',
    data=data
)

plt.title("Fare Distribution by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Fare")

plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = data[['age']]
y = data['fare']

regression_model = LinearRegression()

regression_model.fit(X, y)

y_pred = regression_model.predict(X)

print("Linear Regression Results")
print("-------------------------")
print("Coefficient:", regression_model.coef_[0])
print("Intercept:", regression_model.intercept_)

r2 = r2_score(y, y_pred)

print("R² Score:", r2)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    data['age'],
    data['fare'],
    alpha=0.5
)

plt.plot(
    data['age'],
    y_pred
)

plt.xlabel("Age")
plt.ylabel("Fare")

plt.title("Linear Regression: Age vs Fare")

plt.show()

In [ ]:
X_stats = data[['age']]

X_stats = sm.add_constant(X_stats)

y_stats = data['fare']

model = sm.OLS(y_stats, X_stats).fit()

print(model.summary())

In [ ]:
print("\n========== FINAL CONCLUSION ==========")

if t_pvalue < 0.05:
    print("1. T-Test: There is a statistically significant difference")
    print("   in the average age of survivors and non-survivors.")
else:
    print("1. T-Test: There is no statistically significant difference")
    print("   in the average age of survivors and non-survivors.")

if anova_pvalue < 0.05:
    print("2. ANOVA: Passenger class has a significant effect on fare.")
else:
    print("2. ANOVA: No significant difference in fare was found")
    print("   among passenger classes.")

print(
    "3. Pearson Correlation between Age and Fare:",
    round(correlation, 4)
)

print(
    "4. R² Score of Age vs Fare Regression:",
    round(r2, 4)
)